# Session 14 · Case study: embedding features versus TF-IDF, and nearest-neighbour search

**Block 2 practice.** Compare frozen multilingual sentence embeddings with TF-IDF as classification features on the same subsample, and build a nearest-neighbour search over decisions in several languages.

Needs `sentence-transformers` (first run downloads `intfloat/multilingual-e5-small`, about 470 MB). Encoding 12,000 descriptions takes about 40 seconds with a GPU and several minutes on a CPU; the vectors are cached in `embeddings_cache/` next to this notebook.

Theory: [02-applications-of-embeddings.md](../theory/02-applications-of-embeddings.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
print(decisions.shape)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
year = decisions["start_date"].dt.year
train = decisions[year <= 2021].sample(10_000, random_state=0)      # validation by time
valid = decisions[year >= 2022].sample(2_000, random_state=0)
model = SentenceTransformer(MODEL_NAME)

cache = Path("embeddings_cache")
cache.mkdir(exist_ok=True)


def embed(name, docs, prefix):
    """Encode once, then load from disk; the file name records model, prefix and split."""
    path = cache / f"{MODEL_NAME.split('/')[-1]}_{prefix.strip(': ')}_{name}.npy"
    if path.exists():
        return np.load(path)
    E = model.encode((prefix + docs).tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(path, E)
    return E


E_tr = embed("train", train["description"], "passage: ")
E_va = embed("valid", valid["description"], "query: ")
print(E_tr.shape, E_va.shape)

## 1. Three representations, one classifier

In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score


def svm():
    return SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1)


vec = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(train["description"])
T_tr, T_va = vec.transform(train["description"]), vec.transform(valid["description"])
features = {
    "tf-idf": (T_tr, T_va),
    "embeddings": (E_tr, E_va),
    "both": (hstack([T_tr, csr_matrix(E_tr)]).tocsr(), hstack([T_va, csr_matrix(E_va)]).tocsr()),
}
rows = []
for name, (A_tr, A_va) in features.items():
    pred = svm().fit(A_tr, train["heading"]).predict(A_va)
    rows.append({"features": name, "accuracy": accuracy_score(valid["heading"], pred),
                 "macro_F1": f1_score(valid["heading"], pred, average="macro"),
                 "chapter_accuracy": np.mean(pd.Series(pred).str[:2].to_numpy() == valid["chapter"].to_numpy())})
pd.DataFrame(rows).round(3)

## 2. Learning curve: does the number of labels change the winner?

A 1-nearest-neighbour classifier needs no tuning, so it compares the two representations directly.

In [ ]:
rng = np.random.default_rng(0)
heads, truth = train["heading"].to_numpy(), valid["heading"].to_numpy()
curve = []
for n in [500, 2000, 5000, len(train)]:
    idx = rng.choice(len(train), n, replace=False)
    emb = heads[idx][np.argmax(E_va @ E_tr[idx].T, axis=1)]
    v = TfidfVectorizer(sublinear_tf=True).fit(train["description"].iloc[idx])
    sims = v.transform(valid["description"]) @ v.transform(train["description"].iloc[idx]).T
    tf = heads[idx][np.asarray(sims.argmax(axis=1)).ravel()]
    curve.append({"n_train": n, "embeddings 1-NN": np.mean(emb == truth), "tf-idf 1-NN": np.mean(tf == truth)})
curve = pd.DataFrame(curve).set_index("n_train")
ax = curve.plot(marker="o", logx=True, ylabel="accuracy (validation 2022-2023)", title="Learning curve")
curve.round(3)

**Question.** For English sentiment classification, pre-trained embeddings usually win when only a few hundred labels exist. Why not here? (Hint: how many of the 900 headings have at least one example among 500 training decisions?)

## 3. Nearest-neighbour search across languages

In [ ]:
german = (train["language"] == "de").to_numpy()
corpus = train[german].reset_index(drop=True)
E_de = E_tr[german]
kw = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(corpus["description"])
K = kw.transform(corpus["description"])


def search(query, k=5):
    q = model.encode("query: " + query, normalize_embeddings=True)
    out = []
    for method, scores in [("embedding", E_de @ q),
                           ("tf-idf", (K @ kw.transform([query]).T).toarray().ravel())]:
        for rank, i in enumerate(np.argsort(-scores)[:k], start=1):
            out.append({"method": method, "rank": rank, "score": round(float(scores[i]), 2),
                        "heading": corpus["heading"][i], "keywords": str(corpus["keywords"][i])[:60],
                        "description": corpus["description"][i][:90]})
    return pd.DataFrame(out)


pd.set_option("display.max_colwidth", 100)
search("disposable face masks")

In [ ]:
search("children's bicycle helmet")

The keywords (English, assigned by customs) let you check whether the German hits are relevant.

## 4. Nearest neighbours of a decision

The same index answers "which decisions are most similar to this one?", for example to find decisions of other member states on similar goods.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5, metric="cosine").fit(E_tr)
i = 10
print(valid["language"].iloc[i], valid["heading"].iloc[i], valid["description"].iloc[i][:120], "\n")
dist, ind = nn.kneighbors(E_va[[i]])
for d, j in zip(dist[0], ind[0]):
    print(f"{1 - d:.2f}  {train['language'].iloc[j]}  {train['heading'].iloc[j]}  {train['description'].iloc[j][:90]}")

## Your turn

1. Write three queries of your own in English and one in another language. For each, decide which method gives the more useful top-5, and why.
2. Find a query where TF-IDF is clearly better (hint: brand names, model numbers, CAS numbers of chemicals).
3. Optional: cluster the decisions of one chapter (`train["chapter"] == "95"`) with k-means and cross-tabulate the clusters against `heading` and `language`. Do the clusters reflect goods or languages?
4. Optional: repeat section 1 with `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (no prefixes, 128-token limit) and compare.

In [ ]:
# your code here